# Code for Classification of toric colorable seeds of Picard number $4$
In terms of:
    - Positivity
    - Fan-givingness
    - Projectivity

In [87]:
import SimplicialComplex as sc
import json
import sympy as sp
import numpy as np
from IPython.display import display
from itertools import combinations,permutations

def read_file(filename):
    with open(filename, 'rb') as f:
        data = f.readlines()
        data = [x.strip() for x in data]
    return data

### Function for loading all the seeds of a given pair $(m,n)$

In [88]:
def load_seeds(n,m):
    db_path = 'final_results/CSPLS_%d_%d' % (n, m)
    list_facets = [json.loads(facets_bytes) for facets_bytes in read_file(db_path)]
    return [sc.PureSimplicialComplex(facets_set) for facets_set in list_facets]

def find_orientation(K:sc.PureSimplicialComplex):
    (m,n) = (K.m,K.n)
    orientation = np.zeros(len(K.facets_bin))
    orientation[0]=1
    while np.any(orientation == 0):
        nonoriented = np.where(orientation==0)[0]
        oriented = np.where(orientation!=0)[0]
        for i in nonoriented:
            facet_bin_1 = K.facets_bin[i]
            for j in oriented:
                facet_bin_2 = K.facets_bin[j]
                vertices = sc.binary_to_face_0(facet_bin_1^facet_bin_2,m)
                if len(vertices)==2:
                    vertices = sc.binary_to_face_0(facet_bin_1^facet_bin_2,m)
                    v1=vertices[0]
                    facet1 = sc.binary_to_face_0(facet_bin_1,m)
                    v2=vertices[1]
                    facet2 = sc.binary_to_face_0(facet_bin_2,m)
                    if v1 not in sc.binary_to_face_0(facet_bin_1,m):
                        v1, v2 = v2, v1
                    orientation[i]=(-1)**(facet2.index(v2)+facet1.index(v1)+1)*orientation[j]
                    break
    return(orientation)
    
def compute(K:sc.PureSimplicialComplex,orientation,char_map_skeleton, filter):
    (m,n) = (K.m,K.n)
    # Compute char. map
    location = np.where(filter==False)
    nbr_ind = len(location[0])
    x_vars = sp.symarray('x',nbr_ind)
    poly_char_map = sp.Matrix(char_map_skeleton)
    k=0
    for k in range(nbr_ind):
        poly_char_map[location[0][k],location[1][k]] = x_vars[k]
    N = len(K.facets_bin)
    display(poly_char_map)
    for k in range(1,N):
        rel = poly_char_map[:,sc.binary_to_face_0(K.facets_bin[k],m)].det().expand()-sp.Integer(orientation[k])
        if rel!=0:
            display(rel)
    return poly_char_map

In [95]:

import SimplicialComplex  as sc
dico_seeds ={}
dico_char_maps={}

def fix_ind(char_map_skeleton,filter,i,j,x):
    char_map_skeleton[i,j] = x
    filter[i,j]=True

d_0 = sp.Symbol('d_0')

K_5 = sc.PureSimplicialComplex([[1,2],[1,5],[2,3],[3,4],[4,5]])
dico_seeds[(2,5,0)] = K_5
dico_char_maps[(2,5,0)] = []
dico_char_maps[(2,5,0)].append(sp.Matrix([[1,0,-1,-1,d_0],[0,1,1,0,-1]]))

nbr_of_fangiving={}
nbr_of_fangiving[(2,6)]=1
nbr_of_fangiving[(2,5)]=1

K_6 = sc.PureSimplicialComplex([[1,2],[1,6],[2,3],[3,4],[4,5],[5,6]])
dico_seeds[(2,6,0)] = K_6
dico_char_maps[(2,6,0)] = []
dico_char_maps[(2,6,0)].append(sp.Matrix([[1,0,-1,-1,-1,d_0],[0,1,2,1,0,-1]]))
dico_char_maps[(2,6,0)].append(sp.Matrix([[1,0,-1,2,-1,d_0],[0,1,1,1,0,-1]]))
dico_char_maps[(2,6,0)].append(sp.Matrix([[1,0,-1,-1,d_0-1,d_0],[0,1,1,0,-1,-1]]))

def rec(K):
    (m,n) = (K.m,K.n)
    best_pic=1
    best_v=0
    for v in range(m):
        test = sc.Link_of(K,sc.list_2_pow[v])
        if test.Pic > best_pic and test.is_a_seed():
            best_link = sc.Link_of(K,sc.list_2_pow[v])
            best_pic = best_link.Pic
            best_v=v
    k0=-1
    print(best_pic)
    for k in range(nbr_of_fangiving[(best_link.n,best_link.m)]):
        K=dico_seeds[(best_link.n,best_link.m,k)]
        old_labels = sc.find_isom(K,best_link)
        if old_labels:
            k0=k
            break
    for char_map in dico_char_maps[(best_link.n,best_link.m,k0)]:
            print("hello")
    


for K in load_seeds(3,7):
    rec(K)



4


AttributeError: module 'SimplicialComplex' has no attribute 'find_isom'


2-6 -> 1

3-7 -> 10

4-8 -> 11: [2,3,4,5,6,7,8,9,11,12,14] ->10

5-9 -> 16: [6, 7, 8, 10,  11, 18, 19, 20, 28, 47, 53, 76, 92, 95, 97, 108]

6-10 -> 18: [5,6,11,55,72,126,127,188,271,293,309,326,330,352,392,550,583,616]

7-11 -> 9: [12, 162, 210, 247, 355, 359, 556, 1117, 1155]

8-12 -> 1: [25]

Then -> []
